# MVO4 Swan 8: REPLICATION_GUIDE.md implementation

This revised notebook uses the shared `mvo_algo.py` pipeline. Run from the `replication` directory. The uploaded notebook is preserved separately. The guide controls all dates, swans, and model choices. No live downloads are used.

In [ ]:
import os, sys
from pathlib import Path
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[name] = "1"
HERE = Path.cwd()
if not (HERE / "mvo_algo.py").exists():
    raise RuntimeError("Run this notebook from the replication directory")
sys.path.insert(0, str(HERE))
import pandas as pd
import matplotlib.pyplot as plt
from mvo_algo import Replication, EVENTS, versions, write_outputs
model = Replication(cache=HERE / ".cache/dbi")
versions()

## 1–3. Historical inputs, plain-average crash shape, and inclusive window

In [ ]:
EVENT = 'Swan 8'
K, Seeds, MinAppear, w1 = 7, 3, 3, 0.17
ev = EVENTS[EVENT]
print(ev)
print("Eligible universe:", len(model.training_beta(EVENT)))
print("t1, t2, dd:", model.curve_params(EVENT))
print("start, end, extended:", model.training_window(EVENT))

## 4. Rank all 1,000 seeds, vote, retain the top 22 with at least MinAppear votes

In [ ]:
ranking = model.dbi_ranking(EVENT, K)
print("Best seeds:", ranking[:Seeds])
selected, votes = model.preselect(EVENT, K, Seeds, MinAppear)
display(votes.to_frame())
display(selected)
assert len(selected) >= 5, "This combination has no_preselect status"

## 5–6. Inject at the front, then compute last-year drawdown and weekly covariance ×51

In [ ]:
risk = model.risk_inputs(EVENT, selected)
print("Trading days:", risk["total_days"], "Cutoff:", risk["cutoff"])
pd.Series(risk["curve"], index=risk["raw_prices"].index).plot(title="Injected market drawdown")
plt.show()
display(pd.DataFrame({"fall": risk["fall"], "R_fwd_30_MA": risk["R_fwd"], "E_2": risk["E_2"]}))
display(risk["cov"])

## 7–8. Max Sharpe at rf=0, 3% floor, then buy at peak and evaluate at bottom

In [ ]:
result = model.run_one(EVENT, K, Seeds, MinAppear, w1, detail=True)
print("Status:", result["status"], "PR:", result["PR"])
assert result["status"] == "ok", result.get("error")
display(result["weights"].to_frame("weight"))
pd.DataFrame({"portfolio": result["evaluation"]["portfolio_values"], "benchmark": result["evaluation"]["benchmark_values"]}).plot(title=EVENT)
plt.show()

## 9. Small sweep and optional exports

The command-line runner handles the complete 5,376-run grid. Below is a four-combination event sweep; failed optimization is retained as a status, with no alternate model.

In [ ]:
small = model.run_sweep(dict(K=[7], Seeds=[3], MinAppear=[3], w1=[0.05, 0.11, 0.15, 0.17]), [EVENT])
display(small)
SAVE_OUTPUTS = False
if SAVE_OUTPUTS:
    output = HERE / "results" / EVENT.replace(" ", "_")
    write_outputs(small, output, [EVENT])
    model.export_checkpoints(output / "replication_checkpoints.xlsx", K=K, Seeds=Seeds, MinAppear=MinAppear, w1=w1)

Limitations: original checkpoint workbook is missing; fresh Yahoo downloads and alternative experimental objectives are outside this replication workflow. Swan9 analyst inputs have the guide's point-in-time caveat. See `IMPLEMENTATION_REPORT.md` for validation results.